In [4]:

import requests
import pandas as pd
import sqlite3
import json

API_KEY = "c2b79e13011ff02a5f5ff59ff45689af"

URL = "https://api.themoviedb.org/3/discover/movie"

params = {
    "api_key": API_KEY,
    "language": "en-US",
    "sort_by": "popularity.desc",
    "page": 1
}

response = requests.get(URL, params=params)
data = response.json()

movies = data['results'][:20]

df = pd.DataFrame(movies)

# 🔥 FIX: Convert list columns to string
df['genre_ids'] = df['genre_ids'].apply(lambda x: json.dumps(x))

# Save to SQLite
conn = sqlite3.connect("movies.db")
df.to_sql("movies", conn, if_exists="replace", index=False)
conn.close()



In [5]:
conn = sqlite3.connect("movies.db")
df = pd.read_sql("SELECT * FROM movies", conn)
conn.close()

df['genre_ids'] = df['genre_ids'].apply(lambda x: json.loads(x))

# 1. First 5 rows
print(df.head())

# 2. Summary statistics
print(df.describe())

# 3. Movies per genre
genre_counts = df['genre_ids'].explode().value_counts()
print(genre_counts)

# 4. Missing values
print(df.isnull().sum())

   adult                     backdrop_path      genre_ids       id  \
0      0  /1x9e0qWonw634NhIsRdvnneeqvN.jpg    [10749, 18]  1523145   
1      0  /1fkuBPid72KGS6WmtkEXMftZtkE.jpg       [80, 18]   875828   
2      0  /sdZSjtGUTSN8B3al5o0f2WoQfQQ.jpg  [878, 12, 14]    83533   
3      0  /8Tfys3mDZVp4tNoH2ktm06a0Tau.jpg      [878, 12]   687163   
4      0  /6GuqUJ33BnWDAoVPZP55gDr5G55.jpg   [28, 53, 27]  1084187   

  original_language                    original_title  \
0                ru         Твое сердце будет разбито   
1                en  Peaky Blinders: The Immortal Man   
2                en              Avatar: Fire and Ash   
3                en                 Project Hail Mary   
4                en                     Pretty Lethal   

                                            overview  popularity  \
0  High school student Polina is saved from bully...    543.5230   
1  After his estranged son gets embroiled in a Na...    337.6486   
2  In the wake of the devastatin